# Integrated Gradients

Este notebook:
- Carga un **modelo preentrenado de torchvision** (ResNet50).
- Calcula **Integrated Gradients (IG)** con **Captum** para la **clase predicha** (top-1).
- Procesa **todas** las imágenes de una carpeta y guarda en `output_ig/` un PNG por imagen:
  - Original
  - IG (gris)
  - Heatmap (jet)
  - Overlay


In [1]:
# (Opcional) Instala dependencias si no las tienes
# !pip install -U torch torchvision pillow matplotlib numpy
# !pip install -U captum

from pathlib import Path
import numpy as np
from PIL import Image, UnidentifiedImageError
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120

if torch.cuda.is_available():
    device = torch.device("cuda")      # o "cuda:0"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")       # Apple Silicon
else:
    device = torch.device("cpu")

print("Device:", device)


Device: mps


In [3]:
# Imports Captum

try:
    from captum.attr import IntegratedGradients
except Exception as e:
    raise ImportError("No puedo importar captum. Instala con: pip install captum") from e


In [4]:
# Cargar modelo torchvision (ResNet50 preentrenado)

try:
    from torchvision.models import resnet50, ResNet50_Weights
    weights = ResNet50_Weights.DEFAULT
    model = resnet50(weights=weights)
    imagenet_labels = weights.meta.get("categories", None)
except Exception:
    from torchvision.models import resnet50
    model = resnet50(pretrained=True)
    imagenet_labels = None

model = model.to(device).eval()
ig = IntegratedGradients(model)

print("Modelo listo:", model.__class__.__name__)


Modelo listo: ResNet


In [ ]:
# Utilidades: carga segura + preprocesado ImageNet + overlay

IM_SIZE = 224
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def normalize_0_1(x: np.ndarray, eps=1e-8):
    x = x - x.min()
    x = x / (x.max() + eps)
    return x

def safe_load_rgb(path: str, size=IM_SIZE):
    """Carga imagen (segura) y devuelve:
    - pil_resized
    - img_0_1: HxWx3 float32 [0,1]
    """
    try:
        with Image.open(path) as im:
            im.verify()
        pil = Image.open(path).convert("RGB")
        pil = pil.resize((size, size))
        img_0_1 = np.asarray(pil).astype(np.float32) / 255.0
        return pil, img_0_1
    except (UnidentifiedImageError, OSError) as e:
        print(f"Saltando archivo no válido: {path} | {e}")
        return None, None

def preprocess(img_0_1: np.ndarray) -> torch.Tensor:
    x = (img_0_1 - MEAN) / STD
    x = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0).float()
    x = x.contiguous().to(device) 
    return x

@torch.no_grad()
def predict_proba(img_0_1: np.ndarray) -> np.ndarray:
    x = preprocess(img_0_1)
    logits = model(x)
    probs = F.softmax(logits, dim=1).detach().cpu().numpy()[0]
    return probs

def top1(img_0_1: np.ndarray):
    probs = predict_proba(img_0_1)
    idx = int(np.argmax(probs))
    p = float(probs[idx])
    name = imagenet_labels[idx] if imagenet_labels is not None else str(idx)
    return idx, name, p

def overlay_heatmap_on_image(img_0_1: np.ndarray, m_0_1: np.ndarray, alpha=0.45, cmap_name="jet"):
    m_0_1 = np.clip(m_0_1, 0, 1)
    cmap = plt.get_cmap(cmap_name)
    heat = cmap(m_0_1)[:, :, :3]  # float [0,1]
    overlay = (1 - alpha) * img_0_1 + alpha * heat
    overlay = np.clip(overlay, 0, 1)
    return (heat * 255).astype(np.uint8), (overlay * 255).astype(np.uint8)


In [11]:
# Integrated Gradients map (para la clase objetivo)

def integrated_gradients_map(img_0_1: np.ndarray, target_idx: int, n_steps=32, baseline="black"):
    """Devuelve:
    - base_attr: HxW (float) sin normalizar (puede tener negativos)
    - base_vis: HxW en [0,1] (visualización |abs|)
    """
    x = preprocess(img_0_1)  # 1x3xHxW

    if baseline == "black":
        b = torch.zeros_like(x)
    elif baseline == "mean":
        mean_color = x.mean(dim=(2, 3), keepdim=True)
        b = mean_color.expand_as(x).detach()
    else:
        b = torch.zeros_like(x)

    attr = ig.attribute(x, baselines=b, target=target_idx, n_steps=n_steps)  # 1x3xHxW
    attr = attr.detach().cpu().numpy()[0]  # 3xHxW

    base = attr.sum(axis=0)  # HxW
    base_vis = normalize_0_1(np.abs(base))
    return base, base_vis


In [12]:
# CONFIG: carpeta de entrada y salida
INPUT_DIR = Path("../imagenes/original")    
OUTPUT_DIR = Path("../imagenes/output_ig")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
MAX_IMAGES = None  # None = todas

IG_STEPS = 32
BASELINE = "black"   # "black" o "mean"
USE_POSITIVE_ONLY_FOR_HEATMAP = True  # True = ReLU(base), False = abs(base)

print("INPUT_DIR:", INPUT_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


INPUT_DIR: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/imagenes/original
OUTPUT_DIR: /Users/haojie/PycharmProjects/TFM-Personalized-XAI/imagenes/output_ig


In [13]:
# Ejecutar Integrated Gradients sobre todas las imágenes de la carpeta

paths = [p for p in sorted(INPUT_DIR.rglob("*")) if p.suffix.lower() in EXTS]
if MAX_IMAGES is not None:
    paths = paths[:MAX_IMAGES]

print("Imágenes encontradas:", len(paths))
if len(paths) == 0:
    raise FileNotFoundError(f"No encontré imágenes en {INPUT_DIR}. Revisa INPUT_DIR.")

processed = 0

for i, p in enumerate(paths, 1):
    pil, img_0_1 = safe_load_rgb(str(p), size=IM_SIZE)
    if pil is None:
        continue

    cls_idx, cls_name, cls_prob = top1(img_0_1)

    base, base_vis = integrated_gradients_map(
        img_0_1, target_idx=cls_idx, n_steps=IG_STEPS, baseline=BASELINE
    )

    if USE_POSITIVE_ONLY_FOR_HEATMAP:
        heat_src = normalize_0_1(np.maximum(base, 0))
    else:
        heat_src = normalize_0_1(np.abs(base))

    heat_u8, overlay_u8 = overlay_heatmap_on_image(img_0_1, heat_src, alpha=0.45, cmap_name="jet")

    out_png = OUTPUT_DIR / f"{p.stem}_ig.png"

    fig = plt.figure(figsize=(12, 3))

    ax1 = plt.subplot(1, 4, 1)
    ax1.imshow(img_0_1)
    ax1.set_title("Original")
    ax1.axis("off")

    ax2 = plt.subplot(1, 4, 2)
    ax2.imshow(base_vis, cmap="gray")
    ax2.set_title("IG |abs| (gris)")
    ax2.axis("off")

    ax3 = plt.subplot(1, 4, 3)
    ax3.imshow(heat_u8)
    ax3.set_title("IG heatmap")
    ax3.axis("off")

    ax4 = plt.subplot(1, 4, 4)
    ax4.imshow(overlay_u8)
    ax4.set_title(f"Overlay\n{cls_name} ({cls_prob:.2f})")
    ax4.axis("off")

    plt.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)

    processed += 1
    if processed % 10 == 0 or i == len(paths):
        print(f"Procesadas: {processed} | Último guardado: {out_png}")

print("Revisa la carpeta output_ig/")


Imágenes encontradas: 11
Procesadas: 10 | Último guardado: ../imagenes/output_ig/image10_ig.png
Procesadas: 11 | Último guardado: ../imagenes/output_ig/image11_ig.png
Revisa la carpeta output_ig/
